In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import (
    create_engine, text, Column, Integer, String, Boolean, Text, ForeignKey, CheckConstraint
)
from sqlalchemy.orm import declarative_base, sessionmaker, relationship, joinedload
from contextlib import contextmanager
import pandas as pd

load_dotenv()

True

In [ ]:
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()


@contextmanager
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


In [3]:
from sqlalchemy.exc import OperationalError
# 2. Try to connect and execute
try:
    with engine.connect() as connection:
        # A simple query that works on almost any database
        connection.execute(text("SELECT 1"))
    print("✅ Connection successful!")
except OperationalError as e:
    print(f"❌ Connection failed: {e}")
except Exception as e:
    print(f"⚠️ An unexpected error occurred: {e}")

✅ Connection successful!


In [4]:

with get_db() as db:
    sql = text("SELECT * FROM user_reviews LIMIT(10);")
    data = db.execute(sql)

df = pd.DataFrame(data)
df

,user_id,book_id,book_rating,comment,is_favourite
0,2,25028,NaN,This is a comment,False
1,8,73,5.0,NaN,False
2,8,8211,NaN,NaN,False
3,8,60198,NaN,NaN,False
4,8,71711,NaN,NaN,False
5,8,77832,NaN,NaN,False
6,8,82567,NaN,This is a comment,False
7,8,141577,NaN,NaN,True
8,8,143532,NaN,NaN,True
9,8,157810,5.0,NaN,False


In [5]:
# del Author
# del Publisher
# del User
# del Book
# del UserReview
# del UserReadLater

In [6]:
class Author(Base):
    __tablename__ = "authors"

    id = Column(Integer, primary_key=True)
    name = Column(String(255), nullable=False)

    books = relationship("Book", back_populates="author")


class Publisher(Base):
    __tablename__ = "publishers"

    id = Column(Integer, primary_key=True)
    name = Column(String(255), nullable=False)

    books = relationship("Book", back_populates="publisher")


class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True)
    username = Column(String(255))
    birth_year = Column(Integer)
    email = Column(String(255), unique=True)
    location = Column(String(255))
    user_image = Column(String(2083))

    reviews = relationship("UserReview", back_populates="user")
    readlater = relationship("UserReadLater", back_populates="user")


class Book(Base):
    __tablename__ = "books"

    id = Column(Integer, primary_key=True)
    name = Column(String(1000), nullable=False)
    published_year = Column(Integer)
    image_url = Column(String(2083))
    author_id = Column(Integer, ForeignKey("authors.id"))
    publisher_id = Column(Integer, ForeignKey("publishers.id"))

    author = relationship("Author", back_populates="books")
    publisher = relationship("Publisher", back_populates="books")
    reviews = relationship("UserReview", back_populates="book")
    readlater = relationship("UserReadLater", back_populates="book")


class UserReview(Base):
    __tablename__ = "user_reviews"

    user_id = Column(Integer, ForeignKey("users.id"), primary_key=True)
    book_id = Column(Integer, ForeignKey("books.id"), primary_key=True)
    book_rating = Column(
        Integer,
        CheckConstraint("book_rating BETWEEN 1 AND 10")
    )
    comment = Column(Text)
    is_favourite = Column(Boolean, default=False)

    user = relationship("User", back_populates="reviews")
    book = relationship("Book", back_populates="reviews")


class UserReadLater(Base):
    __tablename__ = "user_readlater"

    user_id = Column(Integer, ForeignKey("users.id"), primary_key=True)
    book_id = Column(Integer, ForeignKey("books.id"), primary_key=True)

    user = relationship("User", back_populates="readlater")
    book = relationship("Book", back_populates="readlater")

In [ ]:
book_id = 55

with get_db() as db:
    book = db.query(Book).options(
        joinedload(Book.reviews).joinedload(UserReview.user),
        joinedload(Book.readlater),
        joinedload(Book.author),
        joinedload(Book.publisher)
    ).filter(Book.id == book_id).first()
    
# with get_db() as db:
#     books = db.query(Book).limit(5)
    
# with get_db() as db:
#     book = db.query(Book).filter(Book.id == book_id).first()

In [71]:
# for book in books:
for review in book.reviews:
    print(review.user.user_image)
    # print(review.user.user_image)

None
None


In [98]:
user_id = 278859
book_id = 137133

with get_db() as db:
    review = db.query(UserReview).filter(
        UserReview.user_id == user_id,
        UserReview.book_id == book_id,
        ).first()

In [100]:
review.is_favourite

False

In [137]:
user_id = 278859
book_id = 137133

with get_db() as db:
    x = db.query(Book).join(UserReview).options(
        joinedload(Book.reviews).joinedload(UserReview.user),
        joinedload(Book.readlater),
        joinedload(Book.author),
        joinedload(Book.publisher)
    ).filter(
        Book.id == book_id,
        UserReview.user_id == user_id
    ).first()

In [138]:
[review.user_id for review in x.reviews]

[99, 113983, 278859]